Sections in the same paper against eachother

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import logging
import json
import os
from sentence_transformers import SentenceTransformer

def extract_text_from_json_files_to_dataframe(directory, section_types):
    logging.basicConfig(level=logging.INFO)
    logging.info(f"Extracting text from JSON files in {directory}")

    data_rows = []

    # Iterate over all JSON files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".json"):
            logging.info(f"Processing file {filename}")
            try:
                with open(os.path.join(directory, filename), 'r') as file:
                    data = json.load(file)
                    base_filename = os.path.splitext(filename)[0]

                    # Extract passages for each section type
                    for section_type in section_types:
                        for passage in data[0]['documents'][0]['passages']:
                            if passage['infons']['section_type'] == section_type:
                                text = passage['text']
                                # Append the extracted text, section type, and filename to the list
                                data_rows.append({
                                    "filename": base_filename,
                                    "section_type": section_type,
                                    "text": text
                                })
            except json.JSONDecodeError as e:
                logging.error(f"Error parsing JSON file {filename}: {e}")
            except KeyError as e:
                logging.error(f"Error extracting text from JSON file {filename}: {e}")

    # Create a DataFrame from the extracted rows
    df = pd.DataFrame(data_rows)
    logging.info(f"Extracted {len(df)} passages")
    return df

def generate_embeddings(df, model_name='all-MiniLM-L6-v2'):
    # Load a pre-trained Sentence Transformer model
    model = SentenceTransformer(model_name)
    
    # Generate embeddings for the 'text' column
    df['embeddings'] = df['text'].apply(lambda x: model.encode(x, show_progress_bar=False))
    
    logging.info("Generated embeddings for all sentences.")
    return df



def compare_sections_pairwise_cosine_similarity(df):
    """
    Compares each pair of sections using pairwise cosine similarity between all sentences in each section.
    """
    section_types = df['section_type'].unique()
    similarities = {}

    # Loop through each pair of section types
    for i, section1 in enumerate(section_types):
        for j, section2 in enumerate(section_types):
            if i >= j:  # Avoid redundant comparisons
                continue

            # Get embeddings for the two sections
            embeddings1 = np.vstack(df[df['section_type'] == section1]['embeddings'].values)
            embeddings2 = np.vstack(df[df['section_type'] == section2]['embeddings'].values)

            # Calculate pairwise cosine similarity for each embedding pair
            pairwise_sim = cosine_similarity(embeddings1, embeddings2)

            # Take the mean, median, or max of the pairwise similarities
            mean_similarity = np.mean(pairwise_sim)
            median_similarity = np.median(pairwise_sim)
            max_similarity = np.max(pairwise_sim)

            # Store the results
            similarities[(section1, section2)] = {
                "mean": mean_similarity,
                "median": median_similarity,
                "max": max_similarity
            }

    return similarities



In [ ]:
if __name__ == "__main__":
    directory = '/Article_folder'
    section_types = ['METHODS', 'RESULTS']
    
    # Extract text and load into DataFrame
    df = extract_text_from_json_files_to_dataframe(directory, section_types)
    
    # Generate embeddings for each sentence
    df_with_embeddings = generate_embeddings(df)
    
    # Save the DataFrame with embeddings (optional)
    df_with_embeddings.to_pickle(os.path.join(directory, 'extracted_data_with_embeddings.pkl'))
    logging.info("Done")


    logging.info("Calculating pairwise cosine similarities for each section.")
    
    # Compute pairwise cosine similarities between sections
    section_similarities = compare_sections_pairwise_cosine_similarity(df_with_embeddings)

    # Display similarities
    for (section1, section2), metrics in section_similarities.items():
        print(f"Comparison between {section1} and {section2}:")
        print(f"  Mean similarity: {metrics['mean']:.4f}")
        print(f"  Median similarity: {metrics['median']:.4f}")
        print(f"  Max similarity: {metrics['max']:.4f}")

In [ ]:
import os
import json
import logging
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from pprint import pprint

def extract_text_from_json_files_to_dataframe(directory, section_types):
    logging.basicConfig(level=logging.INFO)
    logging.info(f"Extracting text from JSON files in {directory}")

    data_rows = []

    # Iterate over all JSON files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".json"):
            logging.info(f"Processing file {filename}")
            try:
                with open(os.path.join(directory, filename), 'r') as file:
                    data = json.load(file)
                    base_filename = os.path.splitext(filename)[0]

                    # Extract passages for each section type
                    for section_type in section_types:
                        for passage in data[0]['documents'][0]['passages']:
                            if passage['infons']['section_type'] == section_type:
                                text = passage['text']
                                # Append the extracted text, section type, and filename to list
                                data_rows.append({
                                    "filename": base_filename,
                                    "section_type": section_type,
                                    "text": text
                                })
            except json.JSONDecodeError as e:
                logging.error(f"Error parsing JSON file {filename}: {e}")
            except KeyError as e:
                logging.error(f"Error extracting text from JSON file {filename}: {e}")

    # Create a DataFrame from extracted rows
    df = pd.DataFrame(data_rows)
    logging.info(f"Extracted {len(df)} passages")
    return df

def generate_embeddings(df, model_name='microsoft/biogpt'):
    # Load model
    model = SentenceTransformer(model_name)
    
    # Generate embeddings for the 'text' column
    df['embeddings'] = df['text'].apply(lambda x: model.encode(x, show_progress_bar=False))
    
    logging.info("Generated embeddings for all sentences.")
    return df

# 1. Average Embedding Comparison
def calculate_average_embeddings(df):
    """
    Calculates the average embedding for each section type in each paper.
    """
    # Group the DataFrame by section_type and filename
    section_groups = df.groupby(['section_type', 'filename'])

    avg_embeddings = {}

    for (section_type, filename), group in section_groups:
        # Get the embeddings for the current section type paper
        embeddings = np.vstack(group['embeddings'].values)

        # Compute the mean embedding
        avg_embedding = np.mean(embeddings, axis=0)

        # Store the average embedding with a combined key of section_type and filename
        avg_embeddings[(section_type, filename)] = avg_embedding

    return avg_embeddings

def compare_avg_embeddings(avg_embeddings):
    """
    Compares the average embeddings between files for the same section type.
    """
    section_types = list(set([key[0] for key in avg_embeddings.keys()]))
    similarities = {}

    # Loop through each section
    for section_type in section_types:
        # Get all papers that contain this section
        filenames = [filename for sec_type, filename in avg_embeddings.keys() if sec_type == section_type]

        # Compare the average embeddings between sections of papers
        for i, file1 in enumerate(filenames):
            for j, file2 in enumerate(filenames):
                if i >= j:
                    continue

                embedding1 = avg_embeddings[(section_type, file1)].reshape(1, -1)
                embedding2 = avg_embeddings[(section_type, file2)].reshape(1, -1)

                # Calculate cosine similarity
                similarity = cosine_similarity(embedding1, embedding2)[0][0]

                similarities[(file1, file2, section_type)] = similarity

    return similarities

# 2. Pairwise Embedding Comparison
def compare_pairwise_embeddings(df):
    """
    Compares each pair of sections between files using pairwise cosine similarity between all sentences.
    """
    section_types = df['section_type'].unique()
    similarities = {}

    # Loop through each section type
    for section_type in section_types:
        # Filter the DataFrame for the current section type
        section_df = df[df['section_type'] == section_type]

        # Get all unique filenames
        filenames = section_df['filename'].unique()

        # Loop through each pair of files for the current section type
        for i, file1 in enumerate(filenames):
            for j, file2 in enumerate(filenames):
                if i >= j:
                    continue

                # Get embeddings for the two files (for this section type)
                embeddings1 = np.vstack(section_df[section_df['filename'] == file1]['embeddings'].values)
                embeddings2 = np.vstack(section_df[section_df['filename'] == file2]['embeddings'].values)

                # Calculate pairwise cosine similarity for each embedding pair
                pairwise_sim = cosine_similarity(embeddings1, embeddings2)

                # Compute statistics: mean, median, and max of the pairwise similarities
                mean_similarity = np.mean(pairwise_sim)
                median_similarity = np.median(pairwise_sim)
                max_similarity = np.max(pairwise_sim)

                # Store the results
                similarities[(file1, file2, section_type)] = {
                    "mean": mean_similarity,
                    "median": median_similarity,
                    "max": max_similarity
                }

    return similarities

# 3. Max Pooling Embedding Comparison
def max_pool_embeddings(embeddings):
    """
    Applies max pooling over a set of embeddings by taking the element-wise maximum.
    """
    return np.max(embeddings, axis=0)

def calculate_max_pooled_embeddings(df):
    """
    Calculates the max pooled embedding for each section of paper.
    """
    section_groups = df.groupby(['section_type', 'filename'])

    max_pooled_embeddings = {}

    for (section_type, filename), group in section_groups:
        # Stack all embeddings for this section and file
        embeddings = np.vstack(group['embeddings'].values)

        # Perform max pooling across all sentence embeddings in the section
        max_pooled_embedding = max_pool_embeddings(embeddings)

        # Store the max pooled embedding
        max_pooled_embeddings[(section_type, filename)] = max_pooled_embedding

    return max_pooled_embeddings

def compare_max_pooled_embeddings(max_pooled_embeddings):
    """
    Compares the max pooled embeddings between files for the same section type.
    """
    section_types = list(set([key[0] for key in max_pooled_embeddings.keys()]))
    similarities = {}

    # Loop through each section type
    for section_type in section_types:
        # Get all filenames that contain this section type
        filenames = [filename for sec_type, filename in max_pooled_embeddings.keys() if sec_type == section_type]

        # Compare the max pooled embeddings between files for this section type
        for i, file1 in enumerate(filenames):
            for j, file2 in enumerate(filenames):
                if i >= j:
                    continue

                embedding1 = max_pooled_embeddings[(section_type, file1)].reshape(1, -1)
                embedding2 = max_pooled_embeddings[(section_type, file2)].reshape(1, -1)

                # Calculate cosine similarity
                similarity = cosine_similarity(embedding1, embedding2)[0][0]

                similarities[(file1, file2, section_type)] = similarity

    return similarities



In [ ]:

if __name__ == "__main__":
    directory = "paper file" 
    section_types = ['METHODS', 'RESULTS']  # Add other sections as needed
    
    # 1: Extract text from JSON files into a DataFrame
    df = extract_text_from_json_files_to_dataframe(directory, section_types)

    # 2: Generate embeddings for each section's text
    df = generate_embeddings(df)

    # Average Embedding Comparison
    avg_embeddings = calculate_average_embeddings(df)
    avg_similarities = compare_avg_embeddings(avg_embeddings)
    print("\nAverage Embedding Similarities:")
    pprint(avg_similarities)

    # Pairwise Embedding Comparison
    pairwise_similarities = compare_pairwise_embeddings(df)
    print("\nPairwise Embedding Similarities:")
    pprint(pairwise_similarities)

    # Max Pooling Embedding Comparison
    max_pooled_embeddings = calculate_max_pooled_embeddings(df)
    max_pooled_similarities = compare_max_pooled_embeddings(max_pooled_embeddings)
    print("\nMax Pooled Embedding Similarities:")
    pprint(max_pooled_similarities)

INFO:root:Extracting text from JSON files in C:/Users/mkha1/Desktop/GQP/Embedding
INFO:root:Processing file 10648562.json
INFO:root:Processing file 10962012.json
INFO:root:Extracted 39 passages
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: microsoft/biogpt
INFO:root:Generated embeddings for all sentences.



Average Embedding Similarities:
{('10648562', '10962012', 'METHODS'): 0.79848754,
 ('10648562', '10962012', 'RESULTS'): 0.69002104}

Pairwise Embedding Similarities:
{('10648562', '10962012', 'METHODS'): {'max': 0.8062962,
                                       'mean': 0.47102156,
                                       'median': 0.44734073},
 ('10648562', '10962012', 'RESULTS'): {'max': 0.827306,
                                       'mean': 0.47420388,
                                       'median': 0.5508731}}

Max Pooled Embedding Similarities:
{('10648562', '10962012', 'METHODS'): 0.83909833,
 ('10648562', '10962012', 'RESULTS'): 0.7904567}
